# Hugging Face DataSets

**Hugging Face Datasets** ist eine Python-Bibliothek aus dem Ökosystem von Hugging Face, die für das effiziente Laden, Verarbeiten und Bereitstellen grosser Datensätze für Machine-Learning- und insbesondere LLM-Anwendungen entwickelt wurde. Sie ist ein zentraler Bestandteil moderner NLP- und LLM-Workflows und wird häufig zusammen mit der Bibliothek Transformers verwendet.

Die Bibliothek stellt eine einheitliche Schnittstelle bereit, um Datensätze aus verschiedenen Quellen zu laden. Dazu gehören öffentliche Datensätze aus dem Hugging Face Hub, lokale Dateien (z. B. JSON, CSV, Parquet, Markdown) oder selbst generierte Trainingsdaten. Ein Datensatz wird intern als tabellarische Struktur gespeichert, die spaltenorientiert aufgebaut ist und effizient auf grossen Datenmengen arbeiten kann.

Technisch basiert die Speicherung auf Apache Arrow. Dieses Format erlaubt speichereffiziente Verarbeitung, Memory-Mapping und schnelles sequentielles Lesen. Dadurch können auch Datensätze mit Millionen von Beispielen verarbeitet werden, ohne vollständig in den Arbeitsspeicher geladen werden zu müssen.

Ein zentrales Konzept der Bibliothek ist das **Dataset-Objekt**. Es repräsentiert eine Sammlung von Datensätzen mit definierten Features (Spalten). Typische Operationen darauf sind:

* Filtern von Beispielen
* Mapping-Funktionen zur Transformation der Daten
* Shuffling und Sampling
* Aufteilen in Trainings-, Validierungs- und Testsets

Ein einfaches Beispiel in Python:

```python
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print(dataset["train"][0])
```

Hier wird automatisch ein Datensatz vom Hugging Face Hub geladen und als strukturierte Dataset-Collection bereitgestellt.

Ein wichtiger Vorteil der Bibliothek ist die **Streaming- und Lazy-Loading-Funktionalität**. Daten können direkt aus dem Internet oder aus grossen lokalen Archiven verarbeitet werden, ohne vollständig heruntergeladen zu werden. Das ist besonders relevant beim Training oder Finetuning grosser Sprachmodelle.

Typische Einsatzgebiete von Hugging Face Datasets sind:

* Training und Finetuning von LLMs
* Aufbau von Evaluations-Datensätzen
* Vorbereitung von Daten für Retrieval-Augmented Generation (RAG)
* Experimentelle Datenanalyse und Preprocessing in Machine-Learning-Pipelines

Zusammengefasst bietet die Bibliothek eine standardisierte und skalierbare Infrastruktur zur Verwaltung von Trainings- und Evaluationsdaten für moderne KI-Workflows.

Erstellen ein Arbeitsverzeichnis `repos` und holen ein paar Repositories.

In [ ]:
%%bash
rm -rf repos
rm -rf dataset
mkdir -p repos
cd repos
git clone https://github.com/mc-b/lerncloud
git clone https://github.com/mc-b/lernmaas


Aufbereitung als DataSet

In [ ]:
from pathlib import Path
from datasets import Dataset
import shutil

# Fixe Verzeichnisse
INPUT_DIR = Path("repos")
OUTPUT_DIR = Path("dataset")

# Alle Markdown-Dateien finden
md_files = list(INPUT_DIR.rglob("*.md"))

rows = []
for file_path in md_files:
    try:
        text = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = file_path.read_text(encoding="utf-8", errors="ignore")

    rows.append({
        "path": str(file_path.relative_to(INPUT_DIR)),
        "source": str(file_path),
        "content": text,
    })

# Hugging Face Dataset erzeugen
ds = Dataset.from_list(rows)

# Output-Verzeichnis neu schreiben
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

ds.save_to_disk(str(OUTPUT_DIR))

print(f"{len(ds)} Markdown-Dateien verarbeitet.")
print(f"Dataset gespeichert unter: {OUTPUT_DIR}")
print(ds)

--- 

## Verwenden des DataSet

DataSet laden

In [ ]:
from datasets import load_from_disk

DATASET_DIR = "dataset"
ds = load_from_disk(DATASET_DIR)

print(ds)
print(ds[0]["path"])

**Embeddings erzeugen**

> Embeddings wandeln Text in eine Zahlenrepräsentation um, sodass ein Computer die Bedeutung von Wörtern und Sätzen vergleichen kann. Dadurch lassen sich Fragen mit inhaltlich passenden Textstellen finden, auch wenn nicht exakt die gleichen Wörter verwendet werden.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

texts = ds["content"]
paths = ds["path"]

embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
print("Embeddings shape:", embeddings.shape)

**Funktionen für Fragen und Antworten**

In [ ]:
def search(question, top_k=3):
    q_emb = model.encode([question], convert_to_numpy=True)[0]

    # Kosinusähnlichkeit
    doc_norms = np.linalg.norm(embeddings, axis=1)
    q_norm = np.linalg.norm(q_emb)
    scores = (embeddings @ q_emb) / (doc_norms * q_norm + 1e-12)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "path": paths[idx],
            "score": float(scores[idx]),
            "content": texts[idx]
        })
    return results

def answer_question(question, top_k=3, max_chars=1200):
    results = search(question, top_k=top_k)

    print(f"Frage: {question}\n")
    print("Relevanteste Treffer:\n")

    for i, r in enumerate(results, 1):
        excerpt = r["content"][:max_chars].strip()
        print(f"[{i}] Datei: {r['path']}")
        print(f"Score: {r['score']:.4f}")
        print(excerpt)
        print("\n" + "-" * 80 + "\n")    

### Fragen 

**Wichtig**: Mit deinem aktuellen Dataset suchst du in ganzen Markdown-Dateien. Das funktioniert für einen ersten Test, aber für bessere Antworten solltest du als Nächstes Chunks erzeugen. Dann findet die Suche gezielter die relevanten Abschnitte statt nur ganze Dokumente.

> Chunks teilen längere Texte in kleinere Abschnitte auf. Dadurch kann ein System gezielter die Textstelle finden, die eine Frage tatsächlich beantwortet.



In [ ]:
answer_question("Voraussetzungen LernMAAS?")

Wenn du statt der ganzen Datei nur eine kurze „Antwort“ ausgeben willst, kannst du noch eine kompaktere Variante verwenden:

In [ ]:
def short_answer(question, top_k=2, max_chars=800):
    results = search(question, top_k=top_k)

    combined = []
    for r in results:
        combined.append(f"Quelle: {r['path']}\n{r['content'][:max_chars].strip()}")

    return "\n\n".join(combined)

In [ ]:
print(short_answer("Voraussetzungen für LernMAAS?"))

In [ ]:
print(short_answer("Projekt LernCloud macht?"))